In [ ]:
!pip install -U langchain langchain-community langchain-core ollama jupyterlab

In [ ]:
# Core LLM + Embeddings
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Document handling (optional for tools)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Python standard library
from typing import List

LLM

In [3]:
llm = ChatOllama(model="llama3", temperature=0)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_5108\1566199341.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0)


Define “tools” as functions

In [4]:
# Calculator tool
def calculator_tool(input: str) -> str:
    try:
        return str(eval(input))
    except Exception as e:
        return f"Error: {e}"

# Lookup tool
knowledge = {
    "LangChain": "A framework for building applications with large language models.",
    "Ollama": "A local LLM server for running models like llama3."
}

def lookup_tool(query: str) -> str:
    return knowledge.get(query, "I don't know.")



Agent Chain

In [5]:
from langchain_core.messages import HumanMessage, AIMessage
import re

chat_history = []

def lookup_tool(query: str) -> str:
    knowledge = {
        "LangChain": "A framework for building applications with large language models.",
        "Ollama": "A local LLM server for running models like llama3."
    }
    # Normalize case
    for key in knowledge:
        if key.lower() in query.lower():
            return knowledge[key]
    return "I don't know."


def calculator_tool(input_text: str) -> str:
    expr = "".join(re.findall(r"[0-9\+\-\*\/\(\)\.]+", input_text))
    if not expr:
        return "No valid math expression found."
    try:
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"

def agent_chain(user_input: str) -> str:
    user_input_lower = user_input.lower()

    if any(c in user_input for c in "+-*/"):
        tool_result = calculator_tool(user_input)
    elif any(word in user_input_lower for word in ["langchain", "ollama"]):
        tool_result = lookup_tool(user_input)
    else:
        tool_result = "I don't know."

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=tool_result))
    return tool_result

In [6]:
agent_chain("What is LangChain?")

'A framework for building applications with large language models.'

In [7]:
agent_chain("Calculate 123 * 45 + 67")

'5602'

In [8]:
agent_chain("Who is Ollama?")

'A local LLM server for running models like llama3.'

In [9]:
agent_chain("What is the capital of Mars?")

"I don't know."